## **Problem 1 : Verify GPU Access**

**Objective:** Confirm that the Colab environment provides a GPU and understand its capabilities.

**Tasks:**

1. Enable GPU runtime in Google Colab.
2. Run:

```python
!nvidia-smi
```

3. Record the GPU model, memory, and CUDA version.

**Reflection Questions:**

- What GPU model was allocated?
- How does the GPU compare to your CPU in terms of cores and memory?

---

## **Problem 2: CPU vs GPU Vector Operations**

**Objective:** Compare simple numerical operations on CPU vs GPU.

**Tasks:**

1. Create two large vectors (e.g., size 20,000,000) using **NumPy**.
2. Perform vector addition on the CPU and measure runtime.
3. Repeat the operation on the GPU using **CuPy**.
4. Compare the execution times.

**Reflection Questions:**

- Which operation was faster and why?
- For smaller arrays, would GPU still be faster?
- How does asynchronous execution affect GPU timing?

---

## **Problem 3 : Measuring Data Transfer Costs** 

**Objective:** Quantify the cost of moving data between CPU and GPU memory.

**Tasks:**

1. Transfer a large array from CPU to GPU.
2. Perform a computation (e.g., multiplication) on the GPU.
3. Transfer the result back to CPU.
4. Measure the runtime of each step separately.

**Reflection Questions:**

- Which step takes the most time?
- How can memory transfer bottlenecks be minimised in GPU programs?
- How does this relate to performance profiling?
---

## **Problem 4: Scaling Matrix Multiplication**

**Objective:** Observe how workload size affects GPU speedup.

**Tasks:**

1. Perform matrix multiplication with increasing sizes: 500×500, 1000×1000, 2000×2000, 4000×4000.
2. Measure runtime on CPU and GPU for each size.
3. Plot a graph comparing CPU vs GPU times.

You may re-use the starter code
```python
# Starter Code for Plotting

# To-do: Add relevant imports


sizes = [500, 1000, 2000, 4000]
cpu_times = []
gpu_times = []

for size in sizes:
    # CPU
    # To-do: 1. random arrays
          #  2. multiplication
          #  3. measure runtime

    # GPU
    # To-do: 1. random arrays
          #  2. multiplication
          #  3. measure runtime

# Plotting Code
plt.plot(sizes, cpu_times, label='CPU')
plt.plot(sizes, gpu_times, label='GPU')
plt.xlabel("Matrix size")
plt.ylabel("Runtime (s)")
plt.title("CPU vs GPU Matrix Multiplication")
plt.legend()
plt.show()
```
---

# Take-Home Assignment
Complete the following challenge at home. Remember to split your screen and solve the problems. You may always Google, check the syntax, resolve errors, and brainstorm, but **do not generate code or answers**. See the [submission instructions](https://github.com/SRH-Heidelberg-University-ADSA/Advanced-Python-for-Data-Science/blob/main/SUBMISSION_INSTRUCTIONS.md). 

# Mini Challenge: Investigating GPU Performance Bottlenecks

## Objective

The objective of this mini challenge is to analyse and compare CPU and GPU execution behaviour across different computational patterns. You will profile several workloads and determine the underlying performance characteristics.

You are expected to identify whether each workload is:

- memory-bound  
- compute-bound  
- dominated by kernel launch overhead  
- affected by repeated CPU-GPU synchronisation  

You should justify all observations using profiling results.

---

## General Instructions

> For all tasks:

1. Measure execution time using `time.perf_counter()`.
2. Ensure GPU correctness by using:

```python
cp.cuda.Stream.null.synchronize()
```

> Before stopping the timer, do the following:

3. Run each experiment at **least three times** and consider the **average value**.
4. Keep array sizes unchanged unless explicitly instructed.
5. Record both CPU and GPU execution times where applicable.

---

## Task 1: Small vs Large Arrays

### Description

This task investigates how input size affects GPU efficiency.

### Program A: Small Scale

```python
import cupy as cp
import time

size = 1_000

a = cp.random.rand(size)
b = cp.random.rand(size)

cp.cuda.Stream.null.synchronize()

start = time.perf_counter()

c = a + b

cp.cuda.Stream.null.synchronize()

end = time.perf_counter()

print("Execution time:", end - start)
```

### Program B: Large Scale

Modify only the following parameter:

```python
size = 100_000_000
```

### Tasks

- Measure execution time for both configurations.
- Compare CPU and GPU execution times.
- Determine the effect of increasing input size on GPU performance.

### Guidance

- Observe whether GPU performance improves with scale.
- Consider kernel launch overhead and parallel execution efficiency.

### Expected Outcome

- Small arrays may show poor GPU utilisation.
- Large arrays should better exploit GPU parallelism.

---

## Task 2: Many Small Kernels

### Description

This task evaluates the cost of repeated kernel launches.

### Program

```python
import cupy as cp
import time

size = 10_000_000

a = cp.random.rand(size)
b = cp.random.rand(size)

cp.cuda.Stream.null.synchronize()

start = time.perf_counter()

for _ in range(1000):
    c = a + b

cp.cuda.Stream.null.synchronize()

end = time.perf_counter()

print("Execution time:", end - start)
```

### Tasks

- Compare this implementation with a single addition operation > ```python c = a + b```
- Measure execution time differences.
- Explain the performance behaviour.

### Guidance

- Each iteration triggers a separate GPU kernel.
- Kernel launch overhead accumulates significantly.

---

## Task 3: Synchronising Too Frequently

### Description

This task examines the impact of repeated synchronisation between CPU and GPU.

### Version A: Single Synchronisation

```python
start = time.perf_counter()

for _ in range(100):
    c = a + b

cp.cuda.Stream.null.synchronize()

end = time.perf_counter()
```

### Version B: Frequent Synchronisation

```python
start = time.perf_counter()

for _ in range(100):
    c = a + b
    cp.cuda.Stream.null.synchronize()

end = time.perf_counter()
```

### Tasks

- Measure execution time for both versions.
- Identify which version is slower.
- Explain the cause of performance degradation.

### Guidance

- Synchronisation forces the CPU to wait for GPU completion.
- It prevents overlap of computation and execution.

---

## Task 4: Memory Transfer Overhead

### Description

This task evaluates the cost of transferring data between CPU and GPU memory.

### CPU to GPU Transfer

```python
import numpy as np
import cupy as cp
import time

size = 10_000_000

a = np.random.rand(size)

start = time.perf_counter()

gpu = cp.asarray(a)

cp.cuda.Stream.null.synchronize()

end = time.perf_counter()

print("CPU to GPU transfer time:", end - start)
```

### GPU to CPU Transfer

```python
start = time.perf_counter()

cpu = cp.asnumpy(gpu)

cp.cuda.Stream.null.synchronize()

end = time.perf_counter()

print("GPU to CPU transfer time:", end - start)
```

### Tasks

- Compare transfer times in both directions.
- Identify which direction is slower, if any.
- Explain why data transfer may become a bottleneck.

### Guidance
- Compare transfer time with computation time.

---

## Task 5: Comparison of GPU Operations

### Description

This task compares different computational kernels with varying arithmetic intensity.

### Operations

```python
c1 = a + b
c2 = cp.sqrt(a)
c3 = cp.exp(a)
C = A @ B
```

### Tasks

- Measure execution time for each operation.
- Rank operations from fastest to slowest.
- Compare CPU vs GPU performance where relevant.

### Guidance

- Element-wise operations are memory-bound.
- Matrix multiplication is compute-bound.
- Non-linear functions may involve higher arithmetic cost.

---

## Final Investigation Table

Complete the table based on your results:

| Experiment | CPU Faster | GPU Faster | Explanation |
|------------|------------|------------|--------------|
| Small array addition | | | Kernel launch overhead dominates |
| Large array addition | | | Increased parallelism |
| Matrix multiplication | | | High compute intensity |
| Frequent synchronisation | | | Blocking behaviour |
| Data transfer | | |Bandwidth limitation |

---

## Reflection Questions

- Why does increasing input size improve GPU utilisation?
- Why does kernel launch overhead affect small workloads disproportionately?
- Why is synchronisation costly in GPU programming?
- Why can data transfer dominate total execution time?
- Which workloads are most suitable for GPU acceleration?
- Under which conditions does the CPU outperform the GPU?
- How does arithmetic intensity influence performance?


### Well done, Week 3 Exercises Complete 🎉!

- Don't forget to submit your [exercise notebook](Week3_Exercise.ipynb) on your private GitHub Repository (see [submission instructions](https://github.com/SRH-Heidelberg-University-ADSA/Advanced-Python-for-Data-Science/blob/main/SUBMISSION_INSTRUCTIONS.md)). The deadline is 05.07.2026 @ 23:59. 

- See you next time when we dive into `Software Design & Testing`!